# 6.7 Decision Trees

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.07_decision-trees.ipynb)

Companion code for Section 6.7 of the book. Run the cells in order from top to bottom.

**Example 6.7.1**

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report
import numpy as np
import matplotlib.pyplot as plt

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      stratify=y, random_state=42)

# ── Effect of max_depth on bias-variance tradeoff ─────────────────
print(f'{'Max Depth':<12} {'Train Acc':>10} {'Test Acc':>10} {'Leaves':>8} {'Verdict'}')
print('='*60)
for depth in [1, 2, 3, 5, 7, 10, None]:
    dt = DecisionTreeClassifier(max_depth=depth, criterion='gini', random_state=42)
    dt.fit(X_train, y_train)
    tr = dt.score(X_train, y_train)
    te = dt.score(X_test, y_test)
    leaves = dt.get_n_leaves()
    verdict = 'UNDERFIT' if te<0.90 else ('OVERFIT' if tr-te>0.05 else 'GOOD')
    depth_str = str(depth) if depth else 'None'
    print(f'{depth_str:<12} {tr:>10.4f} {te:>10.4f} {leaves:>8}  {verdict}')

# ── Best tree with pruning ────────────────────────────────────────
dt_best = DecisionTreeClassifier(max_depth=4, min_samples_split=5,
                                  min_samples_leaf=3, criterion='gini', random_state=42)
dt_best.fit(X_train, y_train)
print(f'\nBest Tree: depth=4, leaves={dt_best.get_n_leaves()}')
print(f'Train: {dt_best.score(X_train, y_train):.4f}  Test: {dt_best.score(X_test, y_test):.4f}')

# ── Feature importance ────────────────────────────────────────────
importance = dt_best.feature_importances_
top_idx = np.argsort(importance)[::-1][:8]
print('\nTop 8 Feature Importances:')
for i in top_idx:
    print(f'  {data.feature_names[i]:30s}: {importance[i]:.4f}')

# ── Print tree rules ──────────────────────────────────────────────
print('\nDecision Tree Rules (depth ≤ 3):')
dt_small = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_small.fit(X_train, y_train)
print(export_text(dt_small, feature_names=list(data.feature_names), max_depth=3))

# ── Visualise tree ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt_small, feature_names=data.feature_names,
         class_names=data.target_names, filled=True, rounded=True,
         fontsize=9, ax=ax)
ax.set_title('Decision Tree (max_depth=3) - Breast Cancer Classification',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Cost-complexity pruning (alpha selection) ─────────────────────
path = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas
train_acc, test_acc = [], []
for alpha in alphas[::5]:  # Sample every 5th alpha
    dt_pruned = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    dt_pruned.fit(X_train, y_train)
    train_acc.append(dt_pruned.score(X_train, y_train))
    test_acc.append(dt_pruned.score(X_test, y_test))

print(f'\nOptimal ccp_alpha: {alphas[::5][np.argmax(test_acc)]:.6f}')
print(f'Test accuracy at optimal pruning: {max(test_acc):.4f}')